In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv('../data/merged_macro_cpi_2000_2026.csv')

# Convert date column to datetime and sort
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y', errors='coerce')
df = df.sort_values('date').reset_index(drop=True)

# Display basic information
print("Dataset shape:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())
print("\nMissing values per column:\n", df.isnull().sum())

# Check data types
print("\nData types:\n", df.dtypes)

# Key columns statistics
key_columns = ['exchange_rate', 'oil_price', 'Food', 'Transport', 'Clothing And Footwear']
print("\nSummary statistics for key columns:\n", df[key_columns].describe())

# Remove any duplicate dates if present
df = df.drop_duplicates(subset=['date'])

print("\nData loading and initial cleaning completed.")
print("Final dataset shape:", df.shape)

Dataset shape: (313, 21)
Date range: 2000-02-01 00:00:00 to 2026-02-01 00:00:00

Missing values per column:
 date                                               0
exchange_rate                                      0
oil_price                                          0
domestic_production                                0
crude_oil_export                                   0
All Items                                          0
All Items Less Farm Produce                        0
All Items Less Farm Produce And Energy             0
Imported Food                                      0
Food                                               0
Alcoholic Beverage Tobacco And Kola                0
Clothing And Footwear                              0
Housing Water Electricity Gas And Other  Fuel      0
Furnishings And Household Equipment Maintenance    0
Health                                             0
Transport                                          0
Communication                              

In [3]:
import pandas as pd

# Load dataset
df = pd.read_csv('../data/merged_macro_cpi_2000_2026.csv')

# Convert date and sort
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y', errors='coerce')
df = df.sort_values('date').reset_index(drop=True)

# Select required columns
required_columns = [
    'date', 'exchange_rate', 'oil_price', 'All Items',
    'Food', 'Transport', 'Clothing And Footwear'
]

df = df[required_columns].copy()

# Calculate MoM % changes
df['all_items_mom'] = df['All Items'].pct_change() * 100
df['food_mom'] = df['Food'].pct_change() * 100
df['transport_mom'] = df['Transport'].pct_change() * 100
df['clothing_mom'] = df['Clothing And Footwear'].pct_change() * 100

# Drop rows with NaN (first row after pct_change)
df = df.dropna().reset_index(drop=True)

# Final information
print("Final dataset shape:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())
print("\nColumns:", df.columns.tolist())
print("\nSummary statistics:\n", df.describe().round(4))

print("\nData preparation completed.")
print("Targets: food_mom, transport_mom, clothing_mom")
print("Key feature added: all_items_mom (headline inflation proxy)")

Final dataset shape: (312, 11)
Date range: 2000-03-01 00:00:00 to 2026-02-01 00:00:00

Columns: ['date', 'exchange_rate', 'oil_price', 'All Items', 'Food', 'Transport', 'Clothing And Footwear', 'all_items_mom', 'food_mom', 'transport_mom', 'clothing_mom']

Summary statistics:
                                 date  exchange_rate  oil_price  All Items  \
count                            312       312.0000   312.0000   312.0000   
mean   2013-02-14 14:27:41.538461440       327.0809    75.1860    46.2561   
min              2000-03-01 00:00:00        99.8700    14.2800     6.8774   
25%              2006-08-24 06:00:00       129.0075    63.8500    16.7622   
50%              2013-02-15 00:00:00       157.3100    66.7100    32.8311   
75%              2019-08-08 18:00:00       307.0000    85.8300    67.7854   
max              2026-02-01 00:00:00      1670.4700   138.7400   147.2893   
std                              NaN       381.3418    22.3703    37.1170   

           Food  Transport  

In [4]:
# Time-based splitting - Recommended professional split

train_end = '2022-12-31'
val_end   = '2024-12-31'

train_df = df[df['date'] <= train_end].copy()
val_df   = df[(df['date'] > train_end) & (df['date'] <= val_end)].copy()
test_df  = df[df['date'] > val_end].copy()

print("Data splitting completed with recommended split.")
print(f"Train set : {train_df.shape[0]:,} rows | {train_df['date'].min().date()} to {train_df['date'].max().date()}")
print(f"Val set   : {val_df.shape[0]:,} rows   | {val_df['date'].min().date()} to {val_df['date'].max().date()}")
print(f"Test set  : {test_df.shape[0]:,} rows  | {test_df['date'].min().date()} to {test_df['date'].max().date()}")

Data splitting completed with recommended split.
Train set : 274 rows | 2000-03-01 to 2022-12-01
Val set   : 24 rows   | 2023-01-01 to 2024-12-01
Test set  : 14 rows  | 2025-01-01 to 2026-02-01


In [5]:
import pandas as pd
import numpy as np

def create_features(data):
    df = data.copy()
    
    # Very lightweight features to maximize recent data retention
    features = ['exchange_rate', 'oil_price', 'all_items_mom']
    lags = [1, 2]                    # Only lag 1 and 2
    windows = [3]                    # Only 3-month rolling
    
    # Lagged features
    for col in features + ['food_mom', 'transport_mom', 'clothing_mom']:
        for lag in lags:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)
    
    # Rolling statistics
    for col in ['exchange_rate', 'oil_price', 'all_items_mom']:
        for window in windows:
            df[f'{col}_roll_mean_{window}'] = df[col].rolling(window=window).mean()
            df[f'{col}_roll_std_{window}'] = df[col].rolling(window=window).std()
    
    # Cyclical month encoding
    df['month'] = df['date'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    # Structural break dummies
    df['post_covid'] = (df['date'] >= '2020-03-01').astype(int)
    df['post_2023_deval'] = (df['date'] >= '2023-06-01').astype(int)
    
    # Forward fill any remaining NaNs (safe for short recent windows)
    df = df.fillna(method='ffill')
    
    # Drop only rows with NaNs at the very beginning
    df = df.dropna().reset_index(drop=True)
    
    return df

# Re-apply lightweight feature engineering
train_df = create_features(train_df)
val_df   = create_features(val_df)
test_df  = create_features(test_df)

print("Lightweight feature engineering applied.")
print(f"Train set after FE: {train_df.shape}")
print(f"Val set after FE:   {val_df.shape}")
print(f"Test set after FE:  {test_df.shape}")

Lightweight feature engineering applied.
Train set after FE: (272, 34)
Val set after FE:   (22, 34)
Test set after FE:  (12, 34)


In [6]:
# Prepare feature matrix (X) and target vectors (y) for each split

# Define targets
targets = ['food_mom', 'transport_mom', 'clothing_mom']

# Feature columns = all columns except date and targets
feature_cols = [col for col in train_df.columns 
                if col not in ['date'] + targets]

print(f"Number of features: {len(feature_cols)}")
print("First 10 features:", feature_cols[:10])

# Create X and y for each set
X_train = train_df[feature_cols].copy()
y_train = train_df[targets].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[targets].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[targets].copy()

print("\nData prepared for modeling:")
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}   | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

Number of features: 30
First 10 features: ['exchange_rate', 'oil_price', 'All Items', 'Food', 'Transport', 'Clothing And Footwear', 'all_items_mom', 'exchange_rate_lag_1', 'exchange_rate_lag_2', 'oil_price_lag_1']

Data prepared for modeling:
X_train shape: (272, 30) | y_train shape: (272, 3)
X_val shape:   (22, 30)   | y_val shape:   (22, 3)
X_test shape:  (12, 30)  | y_test shape:  (12, 3)


In [7]:
import statsmodels.api as sm
import pickle
import os
import warnings
import numpy as np
warnings.filterwarnings('ignore')

# Ensure models directory exists
os.makedirs('../models', exist_ok=True)

targets = ['food_mom', 'transport_mom', 'clothing_mom']
sarimax_models = {}

print("Fitting SARIMAX models with AIC-based selection...\n")

for target in targets:
    print(f"→ Fitting SARIMAX for {target}")
    
    best_aic = np.inf
    best_model = None
    best_order = None
    best_seasonal_order = None
    
    # Small grid search for best orders by AIC
    for p in [1, 2]:
        for q in [1, 2]:
            for P in [0, 1]:
                for Q in [0, 1]:
                    try:
                        model = sm.tsa.SARIMAX(
                            train_df[target],
                            exog=X_train,
                            order=(p, 1, q),
                            seasonal_order=(P, 1, Q, 12),
                            enforce_stationarity=False,
                            enforce_invertibility=False
                        )
                        results = model.fit(disp=False, maxiter=100)
                        
                        if results.aic < best_aic:
                            best_aic = results.aic
                            best_model = results
                            best_order = (p, 1, q)
                            best_seasonal_order = (P, 1, Q, 12)
                            
                    except:
                        continue
    
    sarimax_models[target] = best_model
    
    # Save the best model
    model_path = f'../models/sarimax_{target}.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(best_model, f)
    
    print(f"   Best order: {best_order}, Seasonal: {best_seasonal_order}, AIC: {best_aic:.2f}")
    print(f"   Model saved: {model_path}")

print("\nSARIMAX training completed and models saved to ../models/ folder.")

Fitting SARIMAX models with AIC-based selection...

→ Fitting SARIMAX for food_mom
   Best order: (1, 1, 2), Seasonal: (0, 1, 1, 12), AIC: 612.34
   Model saved: ../models/sarimax_food_mom.pkl
→ Fitting SARIMAX for transport_mom
   Best order: (2, 1, 2), Seasonal: (0, 1, 1, 12), AIC: 1046.03
   Model saved: ../models/sarimax_transport_mom.pkl
→ Fitting SARIMAX for clothing_mom
   Best order: (1, 1, 2), Seasonal: (0, 1, 1, 12), AIC: 944.84
   Model saved: ../models/sarimax_clothing_mom.pkl

SARIMAX training completed and models saved to ../models/ folder.


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import pickle
import os
import warnings
import numpy as np

warnings.filterwarnings('ignore')
os.makedirs('../models', exist_ok=True)

targets = ['food_mom', 'transport_mom', 'clothing_mom']
best_rf_models = {}

print("Tuning separate Random Forest models for each target using TimeSeriesSplit...\n")

for target in targets:
    print(f"→ Tuning Random Forest for {target}")
    
    # Parameter grid
    param_grid = {
        'n_estimators': [200, 300, 400],
        'max_depth': [8, 10, 12],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    
    rf = RandomForestRegressor(random_state=42, n_jobs=-1)
    
    tscv = TimeSeriesSplit(n_splits=5)
    
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=tscv,
        scoring='neg_mean_absolute_error',
        n_jobs=1,
        verbose=0
    )
    
    grid_search.fit(X_train, y_train[target])
    
    best_rf_models[target] = grid_search.best_estimator_
    
    # Save individual tuned model
    model_path = f'../models/randomforest_{target}_tuned.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(grid_search.best_estimator_, f)
    
    print(f"   Best params: {grid_search.best_params_}")
    print(f"   Best CV score (neg MAE): {grid_search.best_score_:.4f}")
    print(f"   Model saved: {model_path}\n")

print("Random Forest tuning completed for all three targets.")
print("Models saved in ../models/ folder:")
for target in targets:
    print(f"   - randomforest_{target}_tuned.pkl")

Tuning separate Random Forest models for each target using TimeSeriesSplit...

→ Tuning Random Forest for food_mom
   Best params: {'max_depth': 12, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 400}
   Best CV score (neg MAE): -0.5830
   Model saved: ../models/randomforest_food_mom_tuned.pkl

→ Tuning Random Forest for transport_mom
   Best params: {'max_depth': 8, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 400}
   Best CV score (neg MAE): -1.4283
   Model saved: ../models/randomforest_transport_mom_tuned.pkl

→ Tuning Random Forest for clothing_mom
   Best params: {'max_depth': 12, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 400}
   Best CV score (neg MAE): -1.0577
   Model saved: ../models/randomforest_clothing_mom_tuned.pkl

Random Forest tuning completed for all three targets.
Models saved in ../models/ folder:
   - randomforest_food_mom_tuned.pkl
   - randomforest_transport_mom_tuned.pkl
   - randomforest_clothing_mom_tuned

In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l1_l2
from sklearn.preprocessing import StandardScaler
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../models', exist_ok=True)

targets = ['food_mom', 'transport_mom', 'clothing_mom']

print("Final LSTM improvement attempt: 2-month window + strong regularization...\n")

for target in targets:
    print(f"→ Training final LSTM for {target}")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    
    # Create sequences with 2-month window
    def create_sequences(X, y, timesteps=2):
        Xs, ys = [], []
        y_series = y[target] if isinstance(y, pd.DataFrame) else y
        for i in range(len(X) - timesteps):
            Xs.append(X[i:(i + timesteps)])
            ys.append(y_series.iloc[i + timesteps])
        return np.array(Xs), np.array(ys)
    
    timesteps = 2
    X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, timesteps)
    X_val_seq,   y_val_seq   = create_sequences(X_val_scaled,   y_val,   timesteps)
    
    print(f"   Sequence shape - Train: {X_train_seq.shape}, Val: {X_val_seq.shape}")
    
    # Very simple + heavily regularized LSTM
    model = Sequential([
        LSTM(24, return_sequences=True, 
             input_shape=(timesteps, X_train.shape[1]),
             kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.4),
        LSTM(12, kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.4),
        Dense(1)
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse', 
        metrics=['mae']
    )
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
    
    history = model.fit(
        X_train_seq, y_train_seq,
        validation_data=(X_val_seq, y_val_seq),
        epochs=100,
        batch_size=32,
        callbacks=[early_stopping, reduce_lr],
        verbose=0
    )
    
    # Save
    model_path = f'../models/lstm_{target}_final.keras'
    model.save(model_path)
    
    scaler_path = f'../models/lstm_scaler_{target}.pkl'
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    
    final_val_mae = history.history['val_mae'][-1]
    
    print(f"   Final validation MAE: {final_val_mae:.4f}")
    print(f"   Model saved: {model_path}")
    print(f"   Scaler saved: {scaler_path}\n")

print("Final LSTM improvement attempt completed.")

Final LSTM improvement attempt: 2-month window + strong regularization...

→ Training final LSTM for food_mom
   Sequence shape - Train: (270, 2, 30), Val: (20, 2, 30)
   Final validation MAE: 3.4138
   Model saved: ../models/lstm_food_mom_final.keras
   Scaler saved: ../models/lstm_scaler_food_mom.pkl

→ Training final LSTM for transport_mom
   Sequence shape - Train: (270, 2, 30), Val: (20, 2, 30)
   Final validation MAE: 3.1146
   Model saved: ../models/lstm_transport_mom_final.keras
   Scaler saved: ../models/lstm_scaler_transport_mom.pkl

→ Training final LSTM for clothing_mom
   Sequence shape - Train: (270, 2, 30), Val: (20, 2, 30)
   Final validation MAE: 1.8478
   Model saved: ../models/lstm_clothing_mom_final.keras
   Scaler saved: ../models/lstm_scaler_clothing_mom.pkl

Final LSTM improvement attempt completed.


In [19]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
import pickle
import os
import warnings
import tensorflow as tf
warnings.filterwarnings('ignore')

os.makedirs('../models', exist_ok=True)

targets = ['food_mom', 'transport_mom', 'clothing_mom']

print("Building Final Stacked Ensemble with ALL three base learners (SARIMAX + RF + LSTM)...\n")

for target in targets:
    print(f"→ Stacking for {target}")
    
    # Load base models
    with open(f'../models/sarimax_{target}.pkl', 'rb') as f:
        sarimax_model = pickle.load(f)
    
    with open(f'../models/randomforest_{target}_tuned.pkl', 'rb') as f:
        rf_model = pickle.load(f)
    
    lstm_model = tf.keras.models.load_model(f'../models/lstm_{target}_final.keras')
    
    # SARIMAX predictions
    train_pred_sar = sarimax_model.predict(exog=X_train, start=0, end=len(X_train)-1)
    val_pred_sar   = sarimax_model.predict(exog=X_val, start=0, end=len(X_val)-1)
    
    # Random Forest predictions
    train_pred_rf = rf_model.predict(X_train)
    val_pred_rf   = rf_model.predict(X_val)
    
    # LSTM predictions
    def get_lstm_predictions(model, X, timesteps=2):
        with open(f'../models/lstm_scaler_{target}.pkl', 'rb') as f:
            scaler = pickle.load(f)
        X_scaled = scaler.transform(X)
        sequences = [X_scaled[i:i+timesteps] for i in range(len(X_scaled) - timesteps)]
        seq_array = np.array(sequences)
        return model.predict(seq_array, verbose=0).flatten()
    
    train_pred_lstm = get_lstm_predictions(lstm_model, X_train)
    val_pred_lstm   = get_lstm_predictions(lstm_model, X_val)
    
    # Align lengths to the shortest
    min_train = min(len(train_pred_sar), len(train_pred_rf), len(train_pred_lstm))
    min_val   = min(len(val_pred_sar), len(val_pred_rf), len(val_pred_lstm))
    
    # Create meta features
    X_train_meta = np.column_stack([
        train_pred_sar[:min_train],
        train_pred_rf[:min_train],
        train_pred_lstm[:min_train]
    ])
    
    X_val_meta = np.column_stack([
        val_pred_sar[:min_val],
        val_pred_rf[:min_val],
        val_pred_lstm[:min_val]
    ])
    
    y_train_meta = y_train[target].iloc[:min_train].values
    y_val_meta   = y_val[target].iloc[:min_val].values
    
    # XGBoost Meta-Learner
    meta_model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='mae'
    )
    
    meta_model.fit(X_train_meta, y_train_meta, verbose=False)
    
    # Save the final stacked model
    meta_path = f'../models/stacked_final_{target}.pkl'
    with open(meta_path, 'wb') as f:
        pickle.dump(meta_model, f)
    
    # Evaluate
    val_pred = meta_model.predict(X_val_meta)
    mae = mean_absolute_error(y_val_meta, val_pred)
    
    print(f"   Validation MAE: {mae:.4f}")
    print(f"   Stacked model saved: {meta_path}\n")

print("✅ Stacked Ensemble with SARIMAX + Random Forest + LSTM completed.")
print("All final stacked models are saved in the ../models/ folder.")

Building Final Stacked Ensemble with ALL three base learners (SARIMAX + RF + LSTM)...

→ Stacking for food_mom
   Validation MAE: 2.9968
   Stacked model saved: ../models/stacked_final_food_mom.pkl

→ Stacking for transport_mom
   Validation MAE: 3.4877
   Stacked model saved: ../models/stacked_final_transport_mom.pkl

→ Stacking for clothing_mom
   Validation MAE: 3.2604
   Stacked model saved: ../models/stacked_final_clothing_mom.pkl

✅ Stacked Ensemble with SARIMAX + Random Forest + LSTM completed.
All final stacked models are saved in the ../models/ folder.


In [20]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle
import os
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

print("=== Overall Model Evaluation on Test Set ===\n")

targets = ['food_mom', 'transport_mom', 'clothing_mom']

evaluation_results = {}

for target in targets:
    print(f"Evaluating {target}...")
    
    # Load final stacked model
    with open(f'../models/stacked_final_{target}.pkl', 'rb') as f:
        stacked_model = pickle.load(f)
    
    # Load base models for meta-feature generation on test set
    with open(f'../models/sarimax_{target}.pkl', 'rb') as f:
        sarimax_model = pickle.load(f)
    with open(f'../models/randomforest_{target}_tuned.pkl', 'rb') as f:
        rf_model = pickle.load(f)
    lstm_model = tf.keras.models.load_model(f'../models/lstm_{target}_final.keras')
    
    # Base predictions on Test set
    test_pred_sar = sarimax_model.predict(exog=X_test, start=0, end=len(X_test)-1)
    test_pred_rf  = rf_model.predict(X_test)
    
    # LSTM prediction on Test
    def get_lstm_pred(X, model, timesteps=2):
        with open(f'../models/lstm_scaler_{target}.pkl', 'rb') as f:
            scaler = pickle.load(f)
        X_scaled = scaler.transform(X)
        sequences = [X_scaled[i:i+timesteps] for i in range(len(X_scaled) - timesteps)]
        seq_array = np.array(sequences)
        return model.predict(seq_array, verbose=0).flatten()
    
    test_pred_lstm = get_lstm_pred(X_test, lstm_model)
    
    # Align lengths
    min_len = min(len(test_pred_sar), len(test_pred_rf), len(test_pred_lstm))
    
    X_test_meta = np.column_stack([
        test_pred_sar[:min_len],
        test_pred_rf[:min_len],
        test_pred_lstm[:min_len]
    ])
    
    y_true = y_test[target].iloc[:min_len].values
    y_pred_stacked = stacked_model.predict(X_test_meta)
    
    # Metrics
    mae = mean_absolute_error(y_true, y_pred_stacked)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_stacked))
    mape = np.mean(np.abs((y_true - y_pred_stacked) / (y_true + 1e-8))) * 100   # avoid division by zero
    
    evaluation_results[target] = {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE (%)': mape
    }
    
    print(f"   MAE : {mae:.4f}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAPE: {mape:.2f}%\n")

print("=== Evaluation Summary ===")
for target, metrics in evaluation_results.items():
    print(f"{target:15} | MAE: {metrics['MAE']:.4f} | RMSE: {metrics['RMSE']:.4f} | MAPE: {metrics['MAPE (%)']:.2f}%")

=== Overall Model Evaluation on Test Set ===

Evaluating food_mom...
   MAE : 1.2326
   RMSE: 1.3993
   MAPE: 138.61%

Evaluating transport_mom...
   MAE : 1.8952
   RMSE: 2.0960
   MAPE: 363.53%

Evaluating clothing_mom...
   MAE : 0.7874
   RMSE: 0.9480
   MAPE: 219.84%

=== Evaluation Summary ===
food_mom        | MAE: 1.2326 | RMSE: 1.3993 | MAPE: 138.61%
transport_mom   | MAE: 1.8952 | RMSE: 2.0960 | MAPE: 363.53%
clothing_mom    | MAE: 0.7874 | RMSE: 0.9480 | MAPE: 219.84%


In [21]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

print("=== Improved Evaluation with sMAPE and MASE ===\n")

targets = ['food_mom', 'transport_mom', 'clothing_mom']

def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

def mase(y_true, y_pred, y_train):
    """Mean Absolute Scaled Error"""
    mae = mean_absolute_error(y_true, y_pred)
    naive_mae = mean_absolute_error(y_true[1:], y_true[:-1])  # naive forecast (previous value)
    return mae / (naive_mae + 1e-8)

for target in targets:
    print(f"Evaluating {target} with better metrics...")
    
    # Load stacked model
    with open(f'../models/stacked_final_{target}.pkl', 'rb') as f:
        stacked_model = pickle.load(f)
    
    # Load base models and generate meta features on test set (same logic as before)
    with open(f'../models/sarimax_{target}.pkl', 'rb') as f:
        sarimax_model = pickle.load(f)
    with open(f'../models/randomforest_{target}_tuned.pkl', 'rb') as f:
        rf_model = pickle.load(f)
    lstm_model = tf.keras.models.load_model(f'../models/lstm_{target}_final.keras')
    
    test_pred_sar = sarimax_model.predict(exog=X_test, start=0, end=len(X_test)-1)
    test_pred_rf  = rf_model.predict(X_test)
    
    def get_lstm_pred(X, model, timesteps=2):
        with open(f'../models/lstm_scaler_{target}.pkl', 'rb') as f:
            scaler = pickle.load(f)
        X_scaled = scaler.transform(X)
        sequences = [X_scaled[i:i+timesteps] for i in range(len(X_scaled) - timesteps)]
        return model.predict(np.array(sequences), verbose=0).flatten()
    
    test_pred_lstm = get_lstm_pred(X_test, lstm_model)
    
    min_len = min(len(test_pred_sar), len(test_pred_rf), len(test_pred_lstm))
    
    X_test_meta = np.column_stack([
        test_pred_sar[:min_len],
        test_pred_rf[:min_len],
        test_pred_lstm[:min_len]
    ])
    
    y_true = y_test[target].iloc[:min_len].values
    y_pred = stacked_model.predict(X_test_meta)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    smape_val = smape(y_true, y_pred)
    mase_val = mase(y_true, y_pred, y_train[target].values)
    
    print(f"   MAE     : {mae:.4f}")
    print(f"   RMSE    : {rmse:.4f}")
    print(f"   sMAPE   : {smape_val:.2f}%")
    print(f"   MASE    : {mase_val:.4f}")
    print("-" * 50)

=== Improved Evaluation with sMAPE and MASE ===

Evaluating food_mom with better metrics...
   MAE     : 1.2326
   RMSE    : 1.3993
   sMAPE   : 100.32%
   MASE    : 1.0743
--------------------------------------------------
Evaluating transport_mom with better metrics...
   MAE     : 1.8952
   RMSE    : 2.0960
   sMAPE   : 128.58%
   MASE    : 1.1326
--------------------------------------------------
Evaluating clothing_mom with better metrics...
   MAE     : 0.7874
   RMSE    : 0.9480
   sMAPE   : 87.39%
   MASE    : 0.6053
--------------------------------------------------


In [9]:
# ==================== FIXED FULL EVALUATION WITH PROPER META-FEATURES ====================
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

def mase(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    naive_mae = np.mean(np.abs(y_true[1:] - y_true[:-1]))
    return mae / (naive_mae + 1e-8) if naive_mae > 0 else np.nan

targets = ['food_mom', 'transport_mom', 'clothing_mom']
target_names = ['Food', 'Transport', 'Clothing & Footwear']

print("🔍 FIXED COMPREHENSIVE EVALUATION WITH PROPER STACKED META-FEATURES")
print("=" * 110)

results = []

for i, target in enumerate(targets):
    print(f"\n📊 Evaluating {target_names[i]}...")
    y_true = y_test[target].values
    
    row = {'Category': target_names[i]}
    
    # ====================== BASE MODEL PREDICTIONS ======================
    
    # SARIMAX
    try:
        with open(f'../models/sarimax_{target}.pkl', 'rb') as f:
            sar_model = pickle.load(f)
        pred_sar = sar_model.predict(exog=X_test, start=0, end=len(X_test)-1)
    except:
        pred_sar = np.full(len(y_true), np.nan)
    
    # Random Forest
    try:
        with open(f'../models/randomforest_{target}_tuned.pkl', 'rb') as f:
            rf_model = pickle.load(f)
        pred_rf = rf_model.predict(X_test)
    except:
        pred_rf = np.full(len(y_true), np.nan)
    
    # LSTM
    try:
        lstm_model = tf.keras.models.load_model(f'../models/lstm_{target}_final.keras')
        with open(f'../models/lstm_scaler_{target}.pkl', 'rb') as f:
            scaler = pickle.load(f)
        X_scaled = scaler.transform(X_test)
        sequences = np.array([X_scaled[j:j+2] for j in range(len(X_scaled)-2)])
        pred_lstm = lstm_model.predict(sequences, verbose=0).flatten()
        min_len = min(len(y_true), len(pred_lstm))
        pred_lstm = np.pad(pred_lstm, (0, len(y_true) - min_len), mode='constant', constant_values=np.nan)
    except:
        pred_lstm = np.full(len(y_true), np.nan)
    
    # ====================== STACKED ENSEMBLE (Meta Features) ======================
    try:
        with open(f'../models/stacked_final_{target}.pkl', 'rb') as f:
            stacked_model = pickle.load(f)
        
        # Create proper meta features
        X_test_meta = np.column_stack([pred_sar, pred_rf, pred_lstm])
        
        pred_stacked = stacked_model.predict(X_test_meta)
        
    except Exception as e:
        print(f"  Stacked model failed: {e}")
        pred_stacked = np.full(len(y_true), np.nan)
    
    # ====================== COMPUTE METRICS ======================
    for name, pred in [("SARIMAX", pred_sar), ("RandomForest", pred_rf), 
                       ("LSTM", pred_lstm), ("Stacked", pred_stacked)]:
        valid = ~np.isnan(pred) & ~np.isnan(y_true)
        if np.sum(valid) > 5:   # Need reasonable number of valid points
            y_t = y_true[valid]
            y_p = pred[valid]
            
            row[f'{name}_MAE'] = mean_absolute_error(y_t, y_p)
            row[f'{name}_RMSE'] = np.sqrt(mean_squared_error(y_t, y_p))
            row[f'{name}_MAPE'] = np.mean(np.abs((y_t - y_p) / (y_t + 1e-8))) * 100
            row[f'{name}_sMAPE'] = smape(y_t, y_p)
            row[f'{name}_MASE'] = mase(y_t, y_p)
        else:
            row[f'{name}_MAE'] = row[f'{name}_RMSE'] = row[f'{name}_MAPE'] = \
            row[f'{name}_sMAPE'] = row[f'{name}_MASE'] = np.nan
    
    results.append(row)

# Display Results
df_eval = pd.DataFrame(results).round(4)

print("\n" + "="*120)
print("FINAL MODEL PERFORMANCE COMPARISON (Test Set)")
print("="*120)
print(df_eval.to_string(index=False))

print("\n🏆 BEST MODEL PER CATEGORY (by MAE):")
for _, row in df_eval.iterrows():
    mae_dict = {
        'SARIMAX': row['SARIMAX_MAE'],
        'Random Forest': row['RandomForest_MAE'],
        'LSTM': row['LSTM_MAE'],
        'Stacked Ensemble': row['Stacked_MAE']
    }
    best = min(mae_dict, key=mae_dict.get)
    print(f"{row['Category']:18} → {best} (MAE: {mae_dict[best]:.4f})")

🔍 FIXED COMPREHENSIVE EVALUATION WITH PROPER STACKED META-FEATURES

📊 Evaluating Food...

📊 Evaluating Transport...

📊 Evaluating Clothing & Footwear...

FINAL MODEL PERFORMANCE COMPARISON (Test Set)
           Category  SARIMAX_MAE  SARIMAX_RMSE  SARIMAX_MAPE  SARIMAX_sMAPE  SARIMAX_MASE  RandomForest_MAE  RandomForest_RMSE  RandomForest_MAPE  RandomForest_sMAPE  RandomForest_MASE  LSTM_MAE  LSTM_RMSE  LSTM_MAPE  LSTM_sMAPE  LSTM_MASE  Stacked_MAE  Stacked_RMSE  Stacked_MAPE  Stacked_sMAPE  Stacked_MASE
               Food       3.5967        4.3256      219.2849       142.0601        1.4826            1.3911             1.7066           113.4574             82.9789             0.5734    1.7424     1.9500   102.6577    185.9223     1.5186       1.5253        1.8597      124.4019        95.6374        0.6288
          Transport       2.9365        5.8362      460.4292       116.0899        1.9034            1.7339             1.9817           294.6278            147.5695             1.